# 00 · Environment smoke test
Run All. Every section prints ✅ or fails with an assertion. This is the acceptance test
for the whole stack: libraries, Java, Spark, the Iceberg REST catalog, MinIO, and PyIceberg.
It creates and then removes its own `env_test` namespace, so it is safe to re-run.

## 1 · Python and libraries

In [ ]:
import importlib, sys

libs = ["numpy", "pandas", "pyarrow", "matplotlib", "seaborn", "plotly",
        "sklearn", "tqdm", "ipywidgets", "findspark", "pyiceberg", "pyspark"]
print("Python", sys.version.split()[0], "\n")
missing = []
for lib in libs:
    try:
        m = importlib.import_module(lib)
        print(f"  ✅ {lib:<12} {getattr(m, '__version__', 'n/a')}")
    except ImportError as e:
        missing.append(lib)
        print(f"  ❌ {lib:<12} {e}")
assert not missing, f"missing libraries: {missing}"

## 2 · Java

In [ ]:
import os, subprocess

r = subprocess.run(["java", "-version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip())
print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
assert r.returncode == 0, "java not found"
print("✅ Java OK")

## 3 · SparkSession

In [ ]:
from pyspark.sql import SparkSession

# Catalog config (REST catalog + MinIO) comes from spark-defaults.conf.
spark = SparkSession.builder.appName("env-test").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

print("✅ Spark", spark.version, "| master:", spark.sparkContext.master)

## 4 · DataFrame and SQL

In [ ]:
from pyspark.sql import functions as F

df = spark.range(100).withColumn("squared", F.col("id") ** 2)
df.createOrReplaceTempView("numbers")
res = spark.sql("SELECT id, squared FROM numbers WHERE id % 10 = 0 ORDER BY id")
res.show()
assert res.count() == 10
print("✅ DataFrame + SQL OK")

## 5 · Spark → pandas (Arrow)

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
pdf = df.limit(5).toPandas()
assert len(pdf) == 5
print(pdf.to_string(index=False))
print("✅ Spark → pandas OK")

## 6 · Iceberg through Spark (REST catalog → MinIO)

In [ ]:
def drop_table(name):
    """Drop an Iceberg table and its data files; fall back to a plain drop."""
    try:
        spark.sql(f"DROP TABLE IF EXISTS {name} PURGE")
    except Exception:
        spark.sql(f"DROP TABLE IF EXISTS {name}")

NS, TBL = "env_test", "numbers"
FULL = f"local.{NS}.{TBL}"

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS local.{NS}")
drop_table(FULL)
df.limit(20).writeTo(FULL).create()

n = spark.table(FULL).count()
assert n == 20, f"expected 20 rows, got {n}"
spark.sql(f"SELECT snapshot_id, operation, committed_at FROM {FULL}.snapshots").show(truncate=False)
print(f"✅ Iceberg write + read OK ({n} rows)")

## 7 · PyIceberg reads the same table

In [ ]:
import os
from pyiceberg.catalog import load_catalog

catalog = load_catalog("rest", **{
    "type": "rest",
    "uri": os.environ["ICEBERG_REST_URI"],
    "s3.endpoint": os.environ["S3_ENDPOINT"],
    "s3.access-key-id": os.environ["AWS_ACCESS_KEY_ID"],
    "s3.secret-access-key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "s3.region": os.environ.get("AWS_REGION", "us-east-1"),
    "s3.force-virtual-addressing": "false",
})
print("namespaces:", catalog.list_namespaces())
arrow_tbl = catalog.load_table(f"{NS}.{TBL}").scan().to_arrow()
assert arrow_tbl.num_rows == 20
print(f"✅ PyIceberg read OK ({arrow_tbl.num_rows} rows)")

## 8 · Clean up the test table

In [ ]:
drop_table(FULL)
spark.sql(f"DROP NAMESPACE IF EXISTS local.{NS}")
print("✅ test table and namespace removed")

## 9 · matplotlib

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x, np.sin(x), label="sin")
ax.plot(x, np.cos(x), linestyle="--", label="cos")
ax.legend()
plt.tight_layout()
plt.show()
print("✅ matplotlib OK")

In [ ]:
print("All checks passed. The environment is ready. 🎉")
spark.stop()